# UCS420: Cognitive Computing
## Assignment 4 - A Cognitive FAQ System Using Pandas (Nova 2.0)


### Q1: Build Your Personalized Knowledge Base
Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:

Take the **LAST TWO DIGITS** of your roll number. For each digit $d$, compute `category = ["billing", "account", "general"][d % 3]`. Invent one realistic question + answer + 3 keywords per entry that fits the assigned category (e.g. if $d \% 3$ gives "account", write a question like "how do I update my registered mobile number").

```python
# Example roll number ...23 -> digits 2, 3
# digit 2 -> category [2 % 3] = general
# digit 3 -> category [3 % 3] = billing
```

**Output:** Print your final 6-row DataFrame.

In [2]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]
categories_list = ["billing", "account", "general"]

roll_number = "1024160079"
req_digits = roll_number[-2:]

req_entries = []
for digit_char in req_digits:
    d = int(digit_char)
    assigned_category = categories_list[d % 3]
    
    if assigned_category == "billing":
        q = {"question": f"how do I check my invoice for digit {d}", "answer": "Invoices are available under the billing tab.", "keywords": "invoice bill charge", "category": assigned_category}
    elif assigned_category == "account":
        q = {"question": f"how do I update my registered mobile number {d}", "answer": "Go to profile settings to update your mobile number.", "keywords": "mobile update profile number", "category": assigned_category}
    else:
        q = {"question": f"where is the helpdesk located for code {d}", "answer": "The main helpdesk is located at Block B.", "keywords": "helpdesk location support office", "category": assigned_category}
        
    req_entries.append(q)

df_faq = pd.DataFrame(fixed_entries + req_entries)
print(df_faq)

                                        question  \
0                         what is the annual fee   
1                          how to reset password   
2                    what are your working hours   
3                          how can i pay the fee   
4  how do I update my registered mobile number 7   
5          how do I check my invoice for digit 9   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to profile settings to update your mobile n...   
5      Invoices are available under the billing tab.   

                       keywords category  
0         fee cost price charge  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4  mobile update profile nu

### Q2: Generate and Score a Hypothesis
Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [3]:
def score_hypothesis(query, df):
    query_words = set(query.lower().split())
    scores = []
    
    for idx, row in df.iterrows():
        keywords = set(str(row['keywords']).lower().split())
        score = len(query_words.intersection(keywords))
        scores.append(score)
        
    df_scored = df.copy()
    df_scored['score'] = scores
    
    return df_scored.sort_values(by='score', ascending=False)

print(score_hypothesis("fee payment", df_faq))

                                        question  \
3                          how can i pay the fee   
0                         what is the annual fee   
1                          how to reset password   
2                    what are your working hours   
4  how do I update my registered mobile number 7   
5          how do I check my invoice for digit 9   

                                              answer  \
3         You can pay via UPI, card, or net banking.   
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
4  Go to profile settings to update your mobile n...   
5      Invoices are available under the billing tab.   

                       keywords category  score  
3           pay payment upi fee  billing      2  
0         fee cost price charge  billing      1  
1          password reset login  account      0  
2        hours timing open time  general  

In [8]:
query_1 = "fee payment"
query_2 = "reset password"

print("Result for Query 1: 'fee payment'")
print(score_hypothesis(query_1, df_faq)[['question', 'category', 'score']])

print("Result for Query 2: 'reset password'")
print(score_hypothesis(query_2, df_faq)[['question', 'category', 'score']])

Result for Query 1: 'fee payment'
                                        question category  score
3                          how can i pay the fee  billing      2
0                         what is the annual fee  billing      1
1                          how to reset password  account      0
2                    what are your working hours  general      0
4  how do I update my registered mobile number 7  account      0
5          how do I check my invoice for digit 9  billing      0
Result for Query 2: 'reset password'
                                        question category  score
1                          how to reset password  account      2
0                         what is the annual fee  billing      0
2                    what are your working hours  general      0
3                          how can i pay the fee  billing      0
4  how do I update my registered mobile number 7  account      0
5          how do I check my invoice for digit 9  billing      0


### Q3: Filter by Category
Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [5]:
def same_category(category_name, df):
    return df[df['category'] == category_name]
target_cat = df_faq.iloc[4]['category']  
print(f"Showing all questions for category: '{target_cat}'\n")
result_q3 = same_category(target_cat, df_faq)
print(result_q3[['question', 'category']])

Showing all questions for category: 'account'

                                        question category
1                          how to reset password  account
4  how do I update my registered mobile number 7  account


### Q4: Update Keywords and Save to CSV
Pick any one entry in your knowledge base[cite: 1]. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named `<your_roll_number>_faq_data.csv`.

In [6]:
target_index = 0
print("Original keywords:", df_faq.loc[target_index, 'keywords'])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

current_keywords = df_faq.loc[target_index, 'keywords']
if new_keyword not in current_keywords:
    df_faq.loc[target_index, 'keywords'] = current_keywords + " " + new_keyword

print("Updated keywords:", df_faq.loc[target_index, 'keywords'])

csv_filename = f"{roll_number}_faq_data.csv"
df_faq.to_csv(csv_filename, index=False)
print(f"DataFrame successfully saved to {csv_filename}")

Original keywords: fee cost price charge


Enter a new keyword to add:  Billing Cost


Updated keywords: fee cost price charge billing cost
DataFrame successfully saved to 1024160079_faq_data.csv


### Q5: Groupby Category Count
Using `groupby`, print how many FAQ entries you have per category.

In [7]:
category_counts = df_faq.groupby('category').size().reset_index(name='count')
print(category_counts)

  category  count
0  account      2
1  billing      3
2  general      1


### Q6: Handle Ties in Scoring
Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one—it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [10]:
def score_hypothesis_with_ties(query, df):
    query_words = set(query.lower().split())
    scores = []
    
    for idx, row in df.iterrows():
        keywords = set(str(row['keywords']).lower().split())
        score = len(query_words.intersection(keywords))
        scores.append(score)
        
    df_scored = df.copy()
    df_scored['score'] = scores
    
    max_score = df_scored['score'].max()
    
    if max_score == 0:
        print(f"Query: '{query}' -> No matching entries found.")
        return pd.DataFrame()
        
    tied_entries = df_scored[df_scored['score'] == max_score]
    
    print(f"Query: '{query}' (Max Score: {max_score})")
    if len(tied_entries) > 1:
        print(f"Note: Found a tie! Displaying all {len(tied_entries)} matching top entries:")
    else:
        print("Top matching entry:")
        
    return tied_entries[['question', 'answer', 'category', 'score']]

tie_query_result = score_hypothesis_with_ties("fee", df_faq)
print(tie_query_result)

no_tie_query_result = score_hypothesis_with_ties("password reset", df_faq)
print(no_tie_query_result)

Query: 'fee' (Max Score: 1)
Note: Found a tie! Displaying all 2 matching top entries:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      1  
3  billing      1  
Query: 'password reset' (Max Score: 2)
Top matching entry:
                question                            answer category  score
1  how to reset password  Go to Settings > Reset Password.  account      2


In [11]:
tie_query = "fee"             
no_tie_query = "working hours"

print("Testing Tie Scenario:")
tie_results = score_hypothesis_with_ties(tie_query, df_faq)
print(tie_results[['question', 'answer', 'category', 'score']])

print("Testing Non-Tie Scenario:")
no_tie_results = score_hypothesis_with_ties(no_tie_query, df_faq)
print(no_tie_results[['question', 'answer', 'category', 'score']])

Testing Tie Scenario:
Query: 'fee' (Max Score: 1)
Note: Found a tie! Displaying all 2 matching top entries:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      1  
3  billing      1  
Testing Non-Tie Scenario:
Query: 'working hours' (Max Score: 1)
Top matching entry:
                      question                     answer category  score
2  what are your working hours  We are open 9 AM to 5 PM.  general      1
